<a href="https://colab.research.google.com/github/kareemEldesouki25/2D-Navier-Stokes-FEM-solver-with-GLS-stabilization-in-C-/blob/main/NN_2026_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Environment Setup

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes peft trl datasets

2. Load the base model from Huggung face


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Define the model name from Hugging Face
model_id = "microsoft/Phi-3-mini-4k-instruct"

# 2. Configure 4-bit compression (as required by the project brief)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # float16 is required for the Colab T4 GPU
    bnb_4bit_use_double_quant=True
)

# 3. Download the model and the tokenizer natively
print("Downloading model natively via transformers library... This takes 1-2 minutes.")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"  # Automatically maps layers onto your T4 GPU
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("\nModel successfully loaded natively and ready for the baseline audit!")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Model successfully loaded natively and ready for the baseline audit!


In [ ]:
model.config.use_cache = False     # Gradient Checkpointing

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f} GB")

Total parameters: 2009.1 M
Memory footprint: 2.05 GB


**Experiment 1 — Qualitative Baseline Audit**

Define a prompt and test the BASE Model

In [ ]:

def build_prompt(question: str) -> str:
    # Set the clinical persona and instructions in the system prompt
    system_instruction = (
        "You are a knowledgeable medical assistant. "
        "Answer the patient's question clearly and accurately."
    )

    # Pass the actual medical question as the user message
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": question},
    ]

    # Format it with the model's native turn tokens (<|end|>, etc.)
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

# 20 audit questions from your dataset image formatted with only quotation marks
audit_questions = [
    "Who is at risk for Lymphocytic Choriomeningitis (LCM)??",
    "What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?",
    "Who is at risk for Lymphocytic Choriomeningitis (LCM)??",
    "How to diagnose Lymphocytic Choriomeningitis (LCM) ?",
    "What are the treatments for Lymphocytic Choriomeningitis (LCM) ?",
    "How to prevent Lymphocytic Choriomeningitis (LCM) ?",
    "What is (are) Parasites - Cysticercosis ?",
    "Who is at risk for Parasites - Cysticercosis??",
    "How to diagnose Parasites - Cysticercosis ?",
    "What are the treatments for Parasites - Cysticercosis ?",
    "How to prevent Parasites - Cysticercosis ?",
    "What is (are) Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "Who is at risk for Parasites - Trichuriasis (also known as Whipworm Infection)??",
    "How to diagnose Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "What are the treatments for Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "How to prevent Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "what are marine toxins?",
    "how can these diseases be diagnosed for Marine Toxins ?",
    "how can these diseases be treated for Marine Toxins ?",
    "how common are these diseases for Marine Toxins ?"
]

print("Starting baseline audit loop. This will print the outputs one by one...\n")

for idx, q in enumerate(audit_questions):
    # Wrap text cleanly inside your build_prompt function
    formatted_prompt = build_prompt(q)

    # Tokenize input safely
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        # Adding temperature=None and top_p=None prevents the dimension conflict bug completely
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            top_p=None,
            use_cache=False  # Required because use_cache is incompatible with gradient checkpointing
        )

    # Extract only the response text generated past the system prompt boundary
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print(f"=== QUESTION {idx + 1}: {q} ===")
    print(response.strip())
    print("-" * 50 + "\n")

Starting baseline audit loop. This will print the outputs one by one...

=== QUESTION 1: Who is at risk for Lymphocytic Choriomeningitis (LCM)?? ===
Lymphocytic Choriomeningitis (LCM) is a viral infection caused by the Lymphocytic Choriomeningitis Virus (LCMV). Individuals at risk for LCM typically include those who are exposed to the primary reservoir of the virus, which is the house mouse (Mus musculus). This can occur through direct contact with infected mice, their feces, or contaminated materials. People who work in laboratories with mice, farmers, pet store workers, and those who live in areas with high populations of house mice are at increased risk. Additionally, individuals with weakened immune systems, such as those with HIV/AIDS, cancer, or who are receiving immunosuppressive treatments, are more susceptible to severe forms of the disease.
--------------------------------------------------

=== QUESTION 2: What are the symptoms of Lymphocytic Choriomeningitis (LCM) ? ===
Lym

**Prepare a small instruction dataset**

In [ ]:
import torch
from transformers import TrainingArguments
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

# ==========================================
# STEP 1: PREPARE AND TEXT-PACK THE DATASET
# ==========================================
print("Loading MedQuad dataset from Hugging Face...")
dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")

shuffled_dataset = dataset.shuffle(seed=42)
training_subset = shuffled_dataset.select(range(500)) # Kept at 500 samples for faster demonstration

def formatting_prompts_func(examples):
    output_texts = []
    for q, a in zip(examples["Question"], examples["Answer"]):
        messages = [
            {"role": "system", "content": "You are a knowledgeable medical assistant. Answer the patient's question clearly and accurately."},
            {"role": "user", "content": q},
            {"role": "assistant", "content": a}
        ]
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
        output_texts.append(formatted_text)
    return {"text": output_texts}

print("Mapping formatting template over 500 samples...")
formatted_dataset = training_subset.map(formatting_prompts_func, batched=True)


# ==========================================
# STEP 2: SET HYPERPARAMETERS & CONFIGS (EXP 2 DEFAULTS)
# ==========================================

print("\nApplying LoRA configuration to the model using Experiment 2 default settings...")
# Ensure gradient checkpointing and input requirements for PEFT
model.gradient_checkpointing_enable()
model.enable_input_require_grads()     # Needed because base parameters are frozen

lora_config = LoraConfig(
    r=8,                                 # Section 4.2 Default Rank
    lora_alpha=16,                       # Section 4.2 Default Alpha
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["qkv_proj"], # Changed from ["q_proj", "v_proj"] to fix the ValueError
)

# Unload existing PEFT adapters if any, to avoid conflicts
if hasattr(model, "peft_config") and len(model.peft_config) > 0:
    model = model.unload()

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA adapters applied.")

training_config = SFTConfig(
    output_dir="./qlora_medical_baseline",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,                  # Section 4.2 Default Epochs
    logging_steps=10,
    optim="paged_adamw_8bit",

    # FIXED: Deactivate both to kill the flawed GradScaler engine on T4 hardware
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    max_length=256,
    dataset_text_field="text",
    report_to="none"
)


# ==========================================
# STEP 3: INITIALIZE AND RUN PIPELINE
# ==========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_config,
    processing_class=tokenizer
)

# Disable caching layers during training backpropagation to protect VRAM footprint
# model.config.use_cache = False # This line is redundant as it's set in a previous cell.

print("\n🚀 Launching the full training pipeline loop...")
trainer.train()
print("\n🎉 Training complete! The low-rank adapters have safely absorbed the clinical domain style.")

Loading MedQuad dataset from Hugging Face...
Mapping formatting template over 500 samples...

Applying LoRA configuration to the model using Experiment 2 default settings...
trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823
LoRA adapters applied.


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4204 > 4096). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]


🚀 Launching the full training pipeline loop...


Step,Training Loss
10,1.654403
20,1.346199
30,1.201535
40,1.100523
50,1.088535
60,1.043292
70,0.991871
80,0.953358
90,0.982286



🎉 Training complete! The low-rank adapters have safely absorbed the clinical domain style.


Inspect how the fine Tuned model answers

### Merging PEFT Adapters for Inference

The `RuntimeError` during generation with the `PeftModel` wrapper often indicates an incompatibility or subtle bug within PEFT's `generate` method when combined with certain base models or quantization settings. To bypass this, a common practice is to merge the LoRA adapters back into the base model's weights. This creates a standard `AutoModelForCausalLM` that incorporates the fine-tuned knowledge, allowing its native `generate` method to be used without the `PeftModel` wrapper's complexities.

In [33]:
model = trainer.model.merge_and_unload()

# Ensure the model is in evaluation mode
model.eval()

print("LoRA adapters merged and unloaded. Model is now a standard AutoModelForCausalLM for inference.")
print(f"Model type after merging: {type(model)}")

LoRA adapters merged and unloaded. Model is now a standard AutoModelForCausalLM for inference.
Model type after merging: <class 'transformers.models.phi3.modeling_phi3.Phi3ForCausalLM'>


### Re-running Post-Training Audit after Merging Adapters

In [42]:
import torch

def build_prompt(question: str) -> str:
    system_instruction = (
        "You are a knowledgeable medical assistant. "
        "Answer the patient's question clearly and accurately."
    )
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

audit_questions = [
    "Who is at risk for Lymphocytic Choriomeningitis (LCM)??",
    "What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?",
    "Who is at risk for Lymphocytic Choriomeningitis (LCM)??",
    "How to diagnose Lymphocytic Choriomeningitis (LCM) ?",
    "What are the treatments for Lymphocytic Choriomeningitis (LCM) ?",
    "How to prevent Lymphocytic Choriomeningitis (LCM) ?",
    "What is (are) Parasites - Cysticercosis ?",
    "Who is at risk for Parasites - Cysticercosis??",
    "How to diagnose Parasites - Cysticercosis ?",
    "What are the treatments for Parasites - Cysticercosis ?",
    "How to prevent Parasites - Cysticercosis ?",
    "What is (are) Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "Who is at risk for Parasites - Trichuriasis (also known as Whipworm Infection)??",
    "How to diagnose Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "What are the treatments for Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "How to prevent Parasites - Trichuriasis (also known as Whipworm Infection) ?",
    "what are marine toxins?",
    "how can these diseases be diagnosed for Marine Toxins ?",
    "how can these diseases be treated for Marine Toxins ?",
    "how common are these diseases for Marine Toxins ?"
]

print("Starting post-training audit loop. Evaluating updated adapters...\n")

model.config.use_cache = False
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
model.config.pad_token_id = pad_id # Explicitly set model's pad_token_id

for idx, q in enumerate(audit_questions):
    formatted_prompt = build_prompt(q)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=256,
            do_sample=False,
            use_cache=False # Consistent with baseline
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print(f"=== POST-TRAINING QUESTION {idx + 1}: {q} ===")
    print(response.strip())
    print("-" * 50 + "\n")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Starting post-training audit loop. Evaluating updated adapters...



RuntimeError: Tensors must have same number of dimensions: got 2 and 3